# Last Assignment: 교육 & 학습 에이전트

## Step 1: 에이전트 설계

### 이름
**PageMate Coach / 영어원서 읽기 코치**

### 목적
영어원서를 읽고 싶지만 책 선정, 분량 조절, 모르는 단어와 숙어 정리 때문에 중간에 포기하는 학습자를 돕는다. 사용자의 영어 수준, 읽고 싶은 책, 학습 목표를 바탕으로 원서 읽기 계획을 만들고, 읽기 전에 익히면 좋은 단어와 숙어 카드를 제공한다.

### 핵심 기능
1. **읽기 목표 정리**: 사용자가 읽고 싶은 책, 현재 영어 수준, 목표, 주간 학습 가능 시간을 정리한다.
2. **원서 읽기 학습계획 생성**: 사용자의 수준과 시간에 맞춰 주차별 읽기 계획을 만든다.
3. **단어 & 숙어 카드 제공**: 원서 읽기에 자주 필요한 단어, 숙어, 예문, 한국어 뜻을 카드 형태로 제공한다.
4. **확장 예정 기능**: 사용자가 읽은 페이지 요약을 입력하면 이해도 점검 퀴즈와 다음 복습 카드를 생성한다.

### 그래프 구조

```text
START
  |
  v
[normalize_profile]
  |  - 책 제목, 수준, 목표, 주간 학습 시간 정리
  v
[build_reading_plan]
  |  - 원서 읽기 학습계획 생성
  v
[create_vocab_cards]
  |  - 단어/숙어 카드 생성
  v
END
```

이번 제출 범위는 Step 2까지이므로, LangGraph로 기본 구조를 만들고 최소 2개 이상의 작동 노드를 구현한다.

## Step 2: 기초 구축

- LangGraph 사용
- 커스텀 State 정의
- 작동 노드 3개 구현
- 기본 그래프 연결 및 실행 예시 포함

아래 코드는 API 키 없이도 실행되도록 규칙 기반으로 작성했다. 이후 LLM을 연결하면 사용자가 선택한 실제 원서의 줄거리, 난이도, 문체에 맞춰 더 정교한 계획과 단어 카드를 생성하도록 확장할 수 있다.

In [ ]:
from typing import TypedDict
from langgraph.graph import START, END, StateGraph


class ReadingCoachState(TypedDict, total=False):
    book_title: str
    english_level: str
    reading_goal: str
    weekly_minutes: int
    reading_plan: list[str]
    vocab_cards: list[dict[str, str]]
    next_action: str

In [ ]:
def normalize_profile(state: ReadingCoachState) -> ReadingCoachState:
    book_title = state.get("book_title", "Charlotte's Web").strip()
    english_level = state.get("english_level", "beginner").strip().lower()
    reading_goal = state.get("reading_goal", "finish the book without giving up").strip()
    weekly_minutes = int(state.get("weekly_minutes", 120))

    return {
        "book_title": book_title,
        "english_level": english_level,
        "reading_goal": reading_goal,
        "weekly_minutes": weekly_minutes,
        "next_action": "Create a reading plan.",
    }


def build_reading_plan(state: ReadingCoachState) -> ReadingCoachState:
    book_title = state["book_title"]
    english_level = state["english_level"]
    reading_goal = state["reading_goal"]
    weekly_minutes = state["weekly_minutes"]

    if english_level in ["beginner", "basic"]:
        plan = [
            f"Week 1: Read short sections of {book_title} for {weekly_minutes // 3} minutes per session and underline repeated words.",
            "Week 2: Summarize each scene in Korean first, then write one simple English sentence.",
            "Week 3: Review vocabulary cards before reading and mark confusing sentences.",
            f"Week 4: Re-read favorite parts and check whether you can {reading_goal}.",
        ]
    else:
        plan = [
            f"Week 1: Read {book_title} in larger chunks and track unknown expressions.",
            "Week 2: Write a short English summary after each reading session.",
            "Week 3: Collect useful phrases and imitate the author's sentence patterns.",
            f"Week 4: Discuss the theme and evaluate whether you can {reading_goal}.",
        ]

    return {"reading_plan": plan, "next_action": "Create vocabulary and phrase cards."}


def create_vocab_cards(state: ReadingCoachState) -> ReadingCoachState:
    book_title = state["book_title"]

    cards = [
        {
            "type": "word",
            "front": "wander",
            "back": "정처 없이 걷다, 돌아다니다",
            "example": f"The character may wander through a new place in {book_title}.",
        },
        {
            "type": "word",
            "front": "hesitate",
            "back": "망설이다",
            "example": "She hesitated before opening the old door.",
        },
        {
            "type": "phrase",
            "front": "make up one's mind",
            "back": "결심하다, 마음을 정하다",
            "example": "He finally made up his mind to keep reading every day.",
        },
        {
            "type": "phrase",
            "front": "out of the blue",
            "back": "갑자기, 예상치 못하게",
            "example": "Out of the blue, an important event changed the story.",
        },
    ]

    return {"vocab_cards": cards, "next_action": "Start Week 1 reading and review the cards first."}

In [ ]:
graph_builder = StateGraph(ReadingCoachState)

graph_builder.add_node("normalize_profile", normalize_profile)
graph_builder.add_node("build_reading_plan", build_reading_plan)
graph_builder.add_node("create_vocab_cards", create_vocab_cards)

graph_builder.add_edge(START, "normalize_profile")
graph_builder.add_edge("normalize_profile", "build_reading_plan")
graph_builder.add_edge("build_reading_plan", "create_vocab_cards")
graph_builder.add_edge("create_vocab_cards", END)

reading_coach = graph_builder.compile()

In [ ]:
result = reading_coach.invoke(
    {
        "book_title": "The Little Prince",
        "english_level": "beginner",
        "reading_goal": "read one chapter without translating every sentence",
        "weekly_minutes": 150,
    }
)

print("Book:", result["book_title"])
print("Level:", result["english_level"])
print("Goal:", result["reading_goal"])

print("\nReading Plan")
for step in result["reading_plan"]:
    print("-", step)

print("\nVocabulary & Phrase Cards")
for index, card in enumerate(result["vocab_cards"], start=1):
    print(f"{index}. [{card['type']}] {card['front']} -> {card['back']}")
    print(f"   Example: {card['example']}")

print("\nNext Action:", result["next_action"])

## 현재 구현 상태

- `ReadingCoachState`: 책 제목, 영어 수준, 읽기 목표, 주간 학습 시간, 학습계획, 단어/숙어 카드를 저장하는 커스텀 State
- `normalize_profile`: 사용자 입력을 정리하고 기본값을 채우는 노드
- `build_reading_plan`: 수준과 목표에 맞는 원서 읽기 계획을 생성하는 노드
- `create_vocab_cards`: 읽기 전에 복습할 단어와 숙어 카드를 생성하는 노드
- `START -> normalize_profile -> build_reading_plan -> create_vocab_cards -> END` 그래프 연결 완료

추후 확장한다면 실제 책 본문 일부를 입력받아 문맥 기반 단어 카드, 이해도 퀴즈, 오답 복습 루프를 추가할 수 있다.